# Weighted Ensemble Submission

This notebook builds a weighted ensemble of anomaly maps from multiple submissions and exports a submission CSV/ZIP.

## 1. Setup

In [1]:
import os
import csv
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

## 2. Configuration

Paths and submission file stems.

In [ ]:
# ---- Paths (edit me) ----
DATASET_ROOT = Path("/kaggle/input/datasets/piants/adl-challenge-anomaly-detection")
SUBMISSIONS_DIR = Path("/kaggle/input/datasets/piants/adl-challenge-submissions/ensemble_test")

# ---- Weighted ensemble ----
PATCHCORE_DINOV2_WEIGHT = 0.1 
PATCHCORE_DINOV3_WEIGHT = 0.15
DESTSEG_WEIGHT = 0.25 
SUPERVISED_WEIGHT = 0.5

SUBMISSION_WEIGHTS = {
    # Keys must match the submission file stem (filename without extension)
    "submission_patchcore": PATCHCORE_DINOV2_WEIGHT,
    "submission_patchcore_dinov3": PATCHCORE_DINOV3_WEIGHT,
    "submission_dino_destseg": DESTSEG_WEIGHT,
    "submission_supervised": SUPERVISED_WEIGHT,
}

OUTPUT_CSV = Path("submission_weighted_ensemble.csv")
OUTPUT_ZIP = Path("submission_weighted_ensemble.zip")
OUTPUT_CSV_PER_CLASS = Path("submission_weighted_ensemble_per-class_norm.csv")
OUTPUT_ZIP_PER_CLASS = Path("submission_weighted_ensemble_per-class_norm.zip")

## 3. Submission utilities

In [8]:
def q8rle_to_float_matrix(s: str) -> np.ndarray:
    t = s.split()
    h, w = int(t[1]), int(t[2])
    vals = np.array(list(map(int, t[3::2])), dtype=np.uint8)
    lens = np.array(list(map(int, t[4::2])), dtype=np.int64)
    flat = np.repeat(vals, lens).reshape(w, h).T
    return flat.astype(np.float32) / 255.0

def float_matrix_to_q8rle(x: np.ndarray) -> str:
    q = np.clip(np.rint(np.asarray(x, dtype=np.float32) * 255), 0, 255).astype(np.uint8)
    h, w = q.shape
    flat = q.T.reshape(-1)
    if flat.size == 0:
        return f"q8rle {h} {w}"
    cuts = np.flatnonzero(flat[1:] != flat[:-1]) + 1
    starts = np.r_[0, cuts]
    ends = np.r_[cuts, flat.size]
    parts = ["q8rle", str(h), str(w)]
    for v, n in zip(flat[starts], ends - starts):
        parts += [str(int(v)), str(int(n))]
    return " ".join(parts)

def write_submission_csv(rows, csv_path):
    path = Path(csv_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["ID", "Label"])
        w.writerows(rows)

def iter_test_images(dataset_dir):
    dataset_dir = Path(dataset_dir)
    for class_dir in sorted(dataset_dir.glob("class_*")):
        test_dir = class_dir / "test"
        for path in sorted(test_dir.rglob("*")):
            if path.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"}:
                yield path.relative_to(dataset_dir).as_posix(), path

def get_original_hw(image_path):
    """(height, width) of the original image on disk."""
    with Image.open(image_path) as im:
        w, h = im.size
    return h, w

def resize_map_to(amap, target_hw):
    """Resize a 2D float anomaly map to target (H, W)."""
    if amap.shape == tuple(target_hw):
        return amap
    t = torch.from_numpy(amap.astype(np.float32))[None, None]
    t = F.interpolate(t, size=tuple(target_hw), mode="bilinear", align_corners=False)
    return t.squeeze().numpy()

def global_normalize_map(amap, gmin, gmax):
    amap = np.asarray(amap, dtype=np.float32)
    if gmax <= gmin:
        return np.zeros_like(amap)
    return (amap - gmin) / (gmax - gmin + 1e-8)

## 4. Load submissions

Loading and preprocessing submissions.

In [9]:
def load_submissions(submissions_dir):
    paths = sorted(list(Path(submissions_dir).rglob("*.csv")) + list(Path(submissions_dir).rglob("*.zip")))
    submissions = {}
    for p in paths:
        df = pd.read_csv(p)
        df = df.set_index("ID")
        submissions[p.stem] = df
    return submissions

submissions_dict = load_submissions(SUBMISSIONS_DIR)
print(f"Loaded {len(submissions_dict)} submissions: {list(submissions_dict.keys())}")

Loaded 5 submissions: ['submission_dino_destseg', 'submission_padim_dinov3', 'submission_patchcore', 'submission_patchcore_dinov3', 'submission_supervised']


In [10]:
keep = ['patchcore', 'patchcore_dinov3', 'supervised', 'dino_destseg']

submissions_dict = {"submission_" + k: submissions_dict["submission_" + k] for k in keep}

In [11]:
def resize_q8rle_map(q8rle):
    # Decode -> [H, W]
    x = q8rle_to_float_matrix(q8rle)

    # Convert to tensor if needed
    if not isinstance(x, torch.Tensor):
        x = torch.tensor(x, dtype=torch.float32)

    # Add batch + channel dimensions
    # [H, W] -> [1, 1, H, W]
    x = x.unsqueeze(0).unsqueeze(0)

    # Resize
    y = F.interpolate(
        x,
        size=(224, 224),
        mode='bilinear',
        align_corners=False
    )

    # Remove batch + channel dims
    # [1,1,224,224] -> [224,224]
    y = y.squeeze(0).squeeze(0)

    # Encode back to q8rle
    return float_matrix_to_q8rle(y)

# Apply to dataframe column
submissions_dict["submission_dino_destseg"]["Label"] = submissions_dict["submission_dino_destseg"]["Label"].apply(resize_q8rle_map)

In [12]:
df = submissions_dict["submission_dino_destseg"]

submissions_dict["submission_dino_destseg"] = df[~df.index.duplicated(keep="first")]

## 5. Weighted ensemble and submission

In [13]:
def weighted_anomaly_map(submissions_dict, weights, sub_id):
    """Sum weighted anomaly maps for a single image ID."""
    acc = None
    for name, w in weights.items():
        df = submissions_dict[name]
        q8rle = df.loc[sub_id, "Label"]
        amap = q8rle_to_float_matrix(q8rle)
        acc = amap * w if acc is None else acc + amap * w
    return acc

print("Step 1: Computing global min/max across all test predictions...")
global_min = float("inf")
global_max = float("-inf")

for img_id, _ in iter_test_images(DATASET_ROOT):
    sub_id = img_id[14:-4]
    amap = weighted_anomaly_map(submissions_dict, SUBMISSION_WEIGHTS, sub_id)
    global_min = min(global_min, float(amap.min()))
    global_max = max(global_max, float(amap.max()))

print(f"Global Min: {global_min:.4f}, Global Max: {global_max:.4f}")
print("Step 2: Resizing, normalizing, and generating RLE submissions...")

rows = []
for img_id, img_path in iter_test_images(DATASET_ROOT):
    sub_id = img_id[14:-4]
    amap = weighted_anomaly_map(submissions_dict, SUBMISSION_WEIGHTS, sub_id)

    orig_hw = get_original_hw(img_path)
    amap = resize_map_to(amap, orig_hw)
    amap = global_normalize_map(amap, global_min, global_max)

    rows.append((sub_id, float_matrix_to_q8rle(amap)))

write_submission_csv(rows, OUTPUT_CSV)
print(f"Wrote {len(rows)} rows to {OUTPUT_CSV}")

Step 1: Computing global min/max across all test predictions...
Global Min: 0.0000, Global Max: 1.0000
Step 2: Resizing, normalizing, and generating RLE submissions...
Wrote 5910 rows to submission_weighted_ensemble.csv


In [14]:
!zip -j {OUTPUT_ZIP} {OUTPUT_CSV}

  adding: submission_weighted_ensemble.csv (deflated 82%)


In [15]:
from IPython.display import FileLink
FileLink(str(OUTPUT_ZIP))

/kaggle/working/submission_weighted_ensemble.zip

In [16]:
# ---- Per-class normalization (weighted ensemble) ----
print("Computing per-class min/max bounds...")
class_minmax = {}
for img_id, _ in iter_test_images(DATASET_ROOT):
    cn = img_id.split("/")[0]
    sub_id = img_id[14:-4]
    amap = weighted_anomaly_map(submissions_dict, SUBMISSION_WEIGHTS, sub_id)
    if cn not in class_minmax:
        class_minmax[cn] = (float(amap.min()), float(amap.max()))
    else:
        cmin, cmax = class_minmax[cn]
        class_minmax[cn] = (min(cmin, float(amap.min())), max(cmax, float(amap.max())))
for cn, (cmin, cmax) in class_minmax.items():
    print(f"  {cn}: min={cmin:.4f}  max={cmax:.4f}")

print("\nResizing, normalizing per class, encoding...")
rows = []
for img_id, img_path in iter_test_images(DATASET_ROOT):
    cn = img_id.split("/")[0]
    sub_id = img_id[14:-4]
    cmin, cmax = class_minmax[cn]
    amap = weighted_anomaly_map(submissions_dict, SUBMISSION_WEIGHTS, sub_id)
    orig_hw = get_original_hw(img_path)
    amap = resize_map_to(amap, orig_hw)
    amap = global_normalize_map(amap, cmin, cmax)
    rows.append((sub_id, float_matrix_to_q8rle(amap)))

write_submission_csv(rows, OUTPUT_CSV_PER_CLASS)
print(f"Wrote {len(rows)} rows to {OUTPUT_CSV_PER_CLASS}")

Computing per-class min/max bounds...
  class_01: min=0.0000  max=1.0000
  class_02: min=0.0000  max=1.0000
  class_03: min=0.0016  max=1.0000
  class_04: min=0.0000  max=1.0000
  class_05: min=0.0000  max=1.0000
  class_06: min=0.0031  max=0.9953
  class_07: min=0.0016  max=1.0000
  class_08: min=0.0000  max=0.9953

Resizing, normalizing per class, encoding...
Wrote 5910 rows to submission_weighted_ensemble_per-class_norm.csv


In [17]:
!zip -j {OUTPUT_ZIP_PER_CLASS} {OUTPUT_CSV_PER_CLASS}

  adding: submission_weighted_ensemble_per-class_norm.csv (deflated 82%)


In [18]:
from IPython.display import FileLink
FileLink(str(OUTPUT_ZIP_PER_CLASS))

/kaggle/working/submission_weighted_ensemble_per-class_norm.zip